# Re-crawl Elsevier DOIs stuck on linkinghub

**Problem**: ~83K Elsevier DOIs from March-April 2025 resolved to `linkinghub.elsevier.com` instead of the actual `sciencedirect.com` article page due to a Taxicab bug that wasn't following JS redirects.

**Solution**: Force re-crawl these DOIs. The bug has been fixed, so they should now resolve correctly.

**How it works**: The Taxicab API always makes fresh Zyte requests (no cache checking), so POSTing the same URLs will fetch new content. We delete the old records from the Databricks table and insert new ones.

**Note**: This creates duplicate records in DynamoDB/R2 (old + new), but that's fine - storage is cheap.

**Created**: 2026-02-02

## Instructions
1. Run cells 1-4 to load the DOIs
2. **TEST FIRST**: Run cell 5 to test with 100 DOIs
3. Check results - verify most resolve to sciencedirect.com
4. If test passes, run cell 6 to process all remaining DOIs
5. Run cells 7-11 to save results

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import concurrent.futures
import datetime
from datetime import timezone
import time
from urllib3.util.retry import Retry
from pyspark.sql import functions as F
from pyspark.sql import types as T
import requests
from requests.adapters import HTTPAdapter
import pandas as pd

In [ ]:
ENDPOINT = "http://harvester-load-balancer-366186003.us-east-1.elb.amazonaws.com/taxicab"

In [ ]:
# Get ALL DOIs that need re-crawling (the full 83K)

linkinghub_dois = spark.sql("""
    SELECT 
        native_id,
        native_id_namespace,
        url,
        resolved_url,
        created_date,
        processed_date,
        taxicab_id
    FROM openalex.taxicab.taxicab_results
    WHERE resolved_url LIKE '%linkinghub.elsevier%'
      AND processed_date BETWEEN '2025-03-01' AND '2025-05-01'
      AND native_id_namespace = 'doi'
""")

total_count = linkinghub_dois.count()
print(f"Found {total_count} DOIs to re-crawl")

In [ ]:
# Convert to pandas - this may take a few minutes for 83K rows
print("Converting to pandas...")
dois_pd = linkinghub_dois.toPandas()
print(f"Converted {len(dois_pd)} rows")

# Build the URL list
jsonUrls = []
for _, row in dois_pd.iterrows():
    entry = {
        "url": row["url"],
        "created_date": row["created_date"] if pd.notna(row["created_date"]) else datetime.datetime.now(timezone.utc),
        "native_id": row["native_id"],
        "native_id_namespace": row["native_id_namespace"],
        "old_taxicab_id": row["taxicab_id"]
    }
    jsonUrls.append(entry)

print(f"Prepared {len(jsonUrls)} URLs for re-crawling")

In [ ]:
# Results schema
results_schema = T.StructType([
    T.StructField("taxicab_id", T.StringType(), True),
    T.StructField("url", T.StringType(), True),
    T.StructField("resolved_url", T.StringType(), True),
    T.StructField("status_code", T.IntegerType(), True),
    T.StructField("content_type", T.StringType(), True),
    T.StructField("native_id", T.StringType(), True),
    T.StructField("native_id_namespace", T.StringType(), True),
    T.StructField("s3_path", T.StringType(), True),
    T.StructField("is_soft_block", T.BooleanType(), True),
    T.StructField("created_date", T.TimestampType(), True),
    T.StructField("processed_date", T.TimestampType(), True),
    T.StructField("error", T.StringType(), True)
])

In [ ]:
def convert_to_datetime(value):
    """Convert various date/time formats to datetime.datetime with UTC timezone"""
    if value is None:
        return datetime.datetime.now(timezone.utc)
    
    if hasattr(value, 'to_pydatetime'):
        dt = value.to_pydatetime()
        if dt.tzinfo is None:
            return dt.replace(tzinfo=timezone.utc)
        return dt
    
    if isinstance(value, datetime.datetime):
        if value.tzinfo is None:
            return value.replace(tzinfo=timezone.utc)
        return value
    
    if isinstance(value, datetime.date):
        return datetime.datetime.combine(value, datetime.time(0, 0, 0, tzinfo=timezone.utc))
    
    return datetime.datetime.now(timezone.utc)

In [ ]:
def process_url_force_recrawl(url_data):
    """
    Process URL - Taxicab always makes fresh Zyte requests (no cache),
    so this will fetch new content even for previously-crawled URLs.
    """
    native_id = url_data.get("native_id", "")
    if native_id and "https://doi.org/" in native_id:
        native_id = native_id.replace("https://doi.org/", "")
    
    try:
        payload = {
            "url": url_data.get("url"),
            "native_id": native_id,
            "native_id_namespace": url_data.get("native_id_namespace", "")
        }
        
        response = requests.post(ENDPOINT, json=payload, timeout=60)
        response.raise_for_status()
        response_data = response.json()
        
        return {
            "taxicab_id": response_data.get("id"),
            "url": url_data.get("url"),
            "status_code": response_data.get("status_code"),
            "resolved_url": response_data.get("resolved_url"),
            "content_type": response_data.get("content_type"),
            "created_date": url_data["created_date"],
            "native_id": response_data.get("native_id"),
            "native_id_namespace": response_data.get("native_id_namespace"),
            "s3_path": response_data.get("s3_path"),
            "is_soft_block": response_data.get("is_soft_block", False),
            "error": None,
            "old_taxicab_id": url_data.get("old_taxicab_id")
        }
    
    except requests.RequestException as e:
        return {
            "taxicab_id": None,
            "url": url_data.get("url"),
            "status_code": getattr(e.response, 'status_code', 0) if hasattr(e, 'response') else 0,
            "resolved_url": None,
            "content_type": None,
            "created_date": url_data["created_date"],
            "native_id": native_id,
            "native_id_namespace": url_data.get("native_id_namespace", ""),
            "s3_path": None,
            "is_soft_block": False,
            "error": str(e),
            "old_taxicab_id": url_data.get("old_taxicab_id")
        }

In [ ]:
def process_urls_with_threadpool(url_list, max_workers=120):
    """
    Process URLs using a ThreadPoolExecutor to parallelize requests.
    """
    results = []
    
    session = requests.Session()
    retries = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )
    adapter = HTTPAdapter(
        pool_connections=120,
        pool_maxsize=120,
        max_retries=retries
    )
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    
    print(f"Starting ThreadPool with {max_workers} workers to process {len(url_list)} URLs")
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(process_url_force_recrawl, url_data): url_data for url_data in url_list}
        
        count = 0
        total = len(future_to_url)
        
        for future in concurrent.futures.as_completed(future_to_url):
            count += 1
            if count % 1000 == 0:
                elapsed = time.time() - start_time
                rate = count / elapsed
                remaining = (total - count) / rate if rate > 0 else 0
                print(f"Processed {count}/{total} ({rate:.1f}/sec, ~{remaining/60:.1f} min remaining)")
            
            url_data = future_to_url[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as exc:
                print(f"URL {url_data.get('url')} generated an exception: {exc}")
                results.append({
                    "taxicab_id": None,
                    "url": url_data.get('url'),
                    "status_code": 0,
                    "resolved_url": None,
                    "content_type": None,
                    "created_date": url_data["created_date"],
                    "native_id": url_data.get("native_id"),
                    "native_id_namespace": url_data.get("native_id_namespace", ""),
                    "s3_path": None,
                    "is_soft_block": False,
                    "error": str(exc),
                    "old_taxicab_id": url_data.get("old_taxicab_id")
                })
    
    elapsed_time = time.time() - start_time
    print(f"ThreadPool processing completed in {elapsed_time:.2f} seconds")
    print(f"Processed {len(results)} URLs at {len(results)/elapsed_time:.1f} URLs/sec")
    
    return results

In [ ]:
# ============================================
# FULL RE-CRAWL - Process all DOIs
# ============================================

print(f"Starting full re-crawl of {len(jsonUrls)} DOIs...")
print(f"Estimated time: ~{len(jsonUrls) / 100 / 60:.0f} minutes at ~100 URLs/sec")

results = process_urls_with_threadpool(jsonUrls, max_workers=120)

processed_date = datetime.datetime.now(timezone.utc)

for result in results:
    result["processed_date"] = processed_date
    if "created_date" in result:
        result["created_date"] = convert_to_datetime(result["created_date"])

print(f"\nCompleted processing {len(results)} URLs")

In [ ]:
# ============================================
# RESULTS SUMMARY
# ============================================

sciencedirect = [r for r in results if r.get("resolved_url") and "sciencedirect" in r.get("resolved_url", "")]
journal_specific = [r for r in results if r.get("resolved_url") and "sciencedirect" not in r.get("resolved_url", "") and "linkinghub" not in r.get("resolved_url", "")]
still_linkinghub = [r for r in results if r.get("resolved_url") and "linkinghub" in r.get("resolved_url", "")]
errors = [r for r in results if r.get("error")]
null_resolved = [r for r in results if not r.get("resolved_url") and not r.get("error")]

print(f"\n{'='*60}")
print(f"RE-CRAWL RESULTS")
print(f"{'='*60}")
print(f"✓ Resolved to sciencedirect.com:    {len(sciencedirect):,} ({100*len(sciencedirect)/len(results):.1f}%)")
print(f"✓ Resolved to journal domains:      {len(journal_specific):,} ({100*len(journal_specific)/len(results):.1f}%)")
print(f"✗ Still linkinghub (no improvement): {len(still_linkinghub):,} ({100*len(still_linkinghub)/len(results):.1f}%)")
print(f"! Errors:                            {len(errors):,} ({100*len(errors)/len(results):.1f}%)")
print(f"? NULL resolved_url:                 {len(null_resolved):,} ({100*len(null_resolved)/len(results):.1f}%)")
print(f"{'='*60}")
print(f"TOTAL SUCCESS (sciencedirect + journal): {len(sciencedirect) + len(journal_specific):,} ({100*(len(sciencedirect) + len(journal_specific))/len(results):.1f}%)")

In [ ]:
# Check results - how many resolved correctly vs still hitting linkinghub?
successful = [r for r in results if r.get("resolved_url") and "sciencedirect" in r.get("resolved_url", "")]
still_linkinghub = [r for r in results if r.get("resolved_url") and "linkinghub" in r.get("resolved_url", "")]
errors = [r for r in results if r.get("error")]

print(f"Results summary:")
print(f"  Resolved to sciencedirect: {len(successful)}")
print(f"  Still linkinghub: {len(still_linkinghub)}")
print(f"  Errors: {len(errors)}")
print(f"  Other: {len(results) - len(successful) - len(still_linkinghub) - len(errors)}")

In [ ]:
# Prepare results for saving (remove old_taxicab_id field)
results_for_save = []
for r in results:
    r_copy = {k: v for k, v in r.items() if k != "old_taxicab_id"}
    results_for_save.append(r_copy)

# Create DataFrame
results_df = spark.createDataFrame(results_for_save, schema=results_schema)
print(f"Created DataFrame with {results_df.count()} records")
results_df.show(5, truncate=False)

In [ ]:
# Delete old records before inserting new ones
# This ensures we don't have duplicate entries

old_taxicab_ids = [r.get("old_taxicab_id") for r in results if r.get("old_taxicab_id")]
print(f"Deleting {len(old_taxicab_ids)} old records...")

# Create temp view with IDs to delete
old_ids_df = spark.createDataFrame([(id,) for id in old_taxicab_ids], ["taxicab_id"])
old_ids_df.createOrReplaceTempView("old_ids_to_delete")

# Delete old records
spark.sql("""
    DELETE FROM openalex.taxicab.taxicab_results 
    WHERE taxicab_id IN (SELECT taxicab_id FROM old_ids_to_delete)
""")

print("Old records deleted.")

In [ ]:
# Save new results
results_df.write.mode("append").format("delta").saveAsTable("openalex.taxicab.taxicab_results")
print(f"Saved {results_df.count()} new records to taxicab_results")

In [ ]:
# Verify: Check how many linkinghub DOIs remain from the problem period
remaining = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM openalex.taxicab.taxicab_results
    WHERE resolved_url LIKE '%linkinghub.elsevier%'
      AND processed_date BETWEEN '2025-03-01' AND '2025-05-01'
""").collect()[0]["cnt"]

print(f"Remaining linkinghub DOIs from problem period: {remaining}")